# When subtracting a count beats adding counts

`census_cover_search.ipynb` answered "males aged 5 to 9" by *adding* five counts.  Some
queries are cheaper the other way round: "males aged **5 and over**" is the male total
**minus** the five counts for ages 0 to 4, which is 7 counts instead of 33.  Each count carries
independent noise, so fewer counts means a tighter posterior.

Genomes in `noisyvalue.dataset.evolve` are signed ({-1, 0, 1}), and `CoverProblem`
scores a -1 as subtracting that count.  This notebook:

1. builds the problem with `prune=False`, because the counts being subtracted (ages 0-4)
   do not overlap the query and would otherwise be dropped;
2. runs the Pareto EA;
3. compares it with the exact optimum from an integer program;
4. shows what the difference does to the posterior.

Requires the Vermont county DHC partitions: `fetch_dhc("county", state="VT")`.

In [1]:
import numpy as np
import pandas as pd
import scipy.optimize as opt
import scipy.sparse as sps

from noisyvalue.census import dhc_2020
from noisyvalue.dataset import cover
from noisyvalue.dataset.evolve import pareto_evolutionary_algorithm
from noisyvalue.sample import sample_noisy_values

## The problem

Query: males of every age except 0-4.  Candidates: the two-way marginals `sex*hispanic`
(gives male total) and `sex*age_38_groups` (gives each age).  Variances come from the
view, as before.

In [2]:
view = dhc_2020.view("county", state="50", county="007")      # Chittenden County, VT
candidates = ["sex*hispanic", "sex*age_38_groups"]
variances = {m: float(view.query(m)["variance"].iloc[0]) for m in candidates}

ages = dhc_2020.axis("age").levels
query = {"sex": ("male",), "age": tuple(a for a in ages if a not in ("0", "1", "2", "3", "4"))}

problem = cover.from_product(dhc_2020, query, measurements=candidates,
                             variances=variances, prune=False)
print(problem)
variances

<CoverProblem: 80 candidate counts over 152 atoms (1,227,744 base cells); query is 587,412 base cells>


{'sex*hispanic': 32.25806427001953, 'sex*age_38_groups': 32.25806427001953}

## A hand-built subtraction

Male total (the two `sex*hispanic` male cells: Hispanic and not) minus each single-year
male age 0-4.

In [3]:
def pick(name, **where):
    return np.array([
        n == name and all(labels[dhc_2020.universe.axis_names.index(a)] in v
                          for a, v in where.items())
        for n, labels in problem.slice_labels], dtype=int)

by_hand = (pick("sex*hispanic", sex=("male",), hispanic=("hispanic", "not_hispanic"))
           - pick("sex*age_38_groups", sex=("male",), age=("0", "1", "2", "3", "4"),
                  hispanic=("total",)))
d = problem.describe(by_hand)
print(len(d["selected"]), "counts; coverage", d["coverage"], "; noise", d["noise"])

7 counts; coverage 0 ; noise 225.80644989013672


Zero coverage error with 7 counts.  Subtracting is exact here because every counted cell
is either in the query or cancelled by an equal subtracted cell.

## The evolutionary search

Same call as before.  The genome is 80 counts long.

In [4]:
fronts, _, _ = pareto_evolutionary_algorithm(
    release=problem, query=None, genome_length=problem.genome_length,
    total_generations=1500, num_parents=60, num_children=60,
    tournament_size=4, archive_size=60, rng=np.random.default_rng(0),
    objectives={"noise": cover.noise_objective, "coverage": cover.coverage_objective})

front = sorted({(i.fitnesses["coverage"], i.fitnesses["noise"]) for i in fronts[-1]})
exact_ea = min((i for i in fronts[-1] if i.fitnesses["coverage"] == 0),
               key=lambda i: i.fitnesses["noise"])
print("front (coverage, noise):", [(c, round(n)) for c, n in front[:4]], "...")
print("best zero-coverage:", problem.describe(exact_ea.genome)["noise"],
      "with", len(problem.describe(exact_ea.genome)["selected"]), "counts")

front (coverage, noise): [(0, 1065), (5292, 1032), (10584, 1000), (15876, 129)] ...
best zero-coverage: 1064.5161209106445 with 33 counts


The EA reaches zero coverage, but through the additive route (33 age counts, noise
about 1065), and never finds the 7-count subtraction at noise 226.  The signed
neighbourhood of that solution is a needle in an 80-dimensional space: all of the 7
entries must be right at once, and a partial subtraction leaves a large coverage error.
So the EA's front is dominated by a point it did not find.

## The exact optimum

With a signed integer genome `g`, the variance is `sum(v * |g|)`, so "zero coverage error
at minimum noise" is a linear integer program: write `g = g+ - g-`, require
`A g = query`, minimise `sum v (g+ + g-)`.

In [5]:
A = sps.csr_array(problem.incidence)
n = problem.n_slices
v = problem.variances
res = opt.milp(
    c=np.concatenate([v, v]),
    constraints=opt.LinearConstraint(sps.hstack([A, -A]).tocsr(), problem.query, problem.query),
    integrality=np.ones(2 * n), bounds=opt.Bounds(0, 1))
g_opt = np.rint(res.x[:n] - res.x[n:]).astype(int)
d = problem.describe(g_opt)
print(res.message)
print(len(d["selected"]), "counts; coverage", d["coverage"], "; noise", d["noise"])
for (name, labels), sign in d["selected"]:
    print(f"{sign:+d}  {name:20s} sex={labels[1]:6s} age={labels[2]:4s} hispanic={labels[3]}")

Optimization terminated successfully. (HiGHS Status 7: Optimal)
7 counts; coverage 0 ; noise 225.80644989013672
+1  sex*hispanic         sex=male   age=total hispanic=not_hispanic
+1  sex*hispanic         sex=male   age=total hispanic=hispanic
-1  sex*age_38_groups    sex=male   age=0    hispanic=total
-1  sex*age_38_groups    sex=male   age=1    hispanic=total
-1  sex*age_38_groups    sex=male   age=2    hispanic=total
-1  sex*age_38_groups    sex=male   age=3    hispanic=total
-1  sex*age_38_groups    sex=male   age=4    hispanic=total


The integer program recovers the hand-built answer: 7 counts, noise 226, about a fifth of the EA's.

## What it does to the posterior

Pull each route's values from the view and sample the signed sum.

In [6]:
axes = list(dhc_2020.universe.axis_names)

def posterior(genome, seed):
    selected = cover.selection_frame(dhc_2020, problem, genome)
    parts = [view.query(m).merge(sel, on=["measurement", *axes])
             for m, sel in selected.groupby("measurement")]
    merged = pd.concat(parts, ignore_index=True)
    total = sum(val * s for val, s in zip(merged["value"], merged["sign"]))
    return sample_noisy_values(total, n=20_000, rng=np.random.default_rng(seed))[0]

rows = []
for label, genome in [("EA (adds 33 counts)", np.array(exact_ea.genome)),
                      ("signed optimum (7 counts)", g_opt)]:
    draws = posterior(genome, seed=1)
    lo, hi = draws.credible_interval(0.95)
    rows.append({"route": label, "mean": draws.draws.mean(), "sd": draws.draws.std(),
                 "95% lower": lo, "95% upper": hi,
                 "predicted sd": np.sqrt(problem.noise(genome))})
pd.DataFrame(rows).round(1)

,route,mean,sd,95% lower,95% upper,predicted sd
0,EA (adds 33 counts),78313.7,32.6,78250.0,78379.0,32.6
1,signed optimum (7 counts),78300.5,13.9,78273.0,78328.0,15.0


Both routes estimate the same quantity, and the signed one has less than half the
standard deviation.  The sampled sd (13.9) is a little under the independence prediction
(15.0) because the view's cells are not exactly independent: they are jointly constrained
by the release's invariants, and the sampler draws them jointly.  `CoverProblem.noise`
(and so the EA's objective and the integer program) assumes independence, which is a
good approximation but not exact.

## Notes

- **Search limits.** `pareto_evolutionary_algorithm` starts from dense random genomes and
  mutates one entry at a time.  It found the additive solution here but not the signed
  one; a sparse initialisation did not help in the trials I ran.  Treat the EA's front as an
  upper bound on achievable noise, and use the integer program as a check where the
  problem is small enough.
- The integer program assumes the same independence of noise across counts as
  `CoverProblem.noise`, and it only handles the exact-cover (zero coverage error) case.
  Trading coverage for noise is what the EA's front is for.